<a href="https://colab.research.google.com/github/JoshuneArriaga/Datos-Masivos/blob/main/Tarea_3_Datos_Masivos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalación del entorno Spark en Google Colab

In [1]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://archive.apache.org/dist/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j
!pip install -q findspark pyspark py4j mwparserfromhell
import os
import sys

import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:11 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,793 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,613 kB]
Get:14

In [2]:
spark = SparkSession.builder \
    .appName("Wikipedia DataFrames") \
    .getOrCreate()

spark

## Parsing del XML y conversión a RDD
Se descargan 3 partes del dump de la pagina de wikipedia

In [3]:
!mkdir -p data/wikipedia_src
urls = [
"https://dumps.wikimedia.org/enwiki/20260201/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2",
"https://dumps.wikimedia.org/enwiki/20260201/enwiki-20260201-pages-articles-multistream2.xml-p41243p151573.bz2",
"https://dumps.wikimedia.org/enwiki/20260201/enwiki-20260201-pages-articles-multistream3.xml-p151574p311329.bz2"
]

for url in urls:
    !wget -nc {url} -P data/wikipedia_src

--2026-03-02 05:36:54--  https://dumps.wikimedia.org/enwiki/20260201/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.71, 2620:0:861:3:208:80:154:71
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.71|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 295167616 (281M) [application/octet-stream]
Saving to: ‘data/wikipedia_src/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2’

enwiki-20260201-pag 100%[===================>] 281.49M  4.49MB/s    in 69s     

2026-03-02 05:38:03 (4.09 MB/s) - ‘data/wikipedia_src/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2’ saved [295167616/295167616]

--2026-03-02 05:38:03--  https://dumps.wikimedia.org/enwiki/20260201/enwiki-20260201-pages-articles-multistream2.xml-p41243p151573.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.71, 2620:0:861:3:208:80:154:71
Connecting to dumps.wikimedia.org

In [6]:
import glob

dump_files = sorted(
    glob.glob("data/wikipedia_src/*.bz2")
)

print("Archivos encontrados:")
for f in dump_files:
    print(f)

Archivos encontrados:
data/wikipedia_src/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2
data/wikipedia_src/enwiki-20260201-pages-articles-multistream2.xml-p41243p151573.bz2
data/wikipedia_src/enwiki-20260201-pages-articles-multistream3.xml-p151574p311329.bz2


In [9]:
import bz2
import xml.etree.ElementTree as ET
import mwparserfromhell
import re
import time

def detect_namespace(filepath):
    """Detecta el namespace XML del dump automáticamente."""
    with bz2.open(filepath, 'rt', encoding='utf-8') as f:
        for line in f:
            match = re.search(r'xmlns="([^"]+)"', line)
            if match:
                ns = '{' + match.group(1) + '}'
                print(f'Namespace: {ns}')
                return ns
    return ''

def wikitext_to_plain(wikicode):
    try:
        return mwparserfromhell.parse(wikicode).strip_code().strip()
    except:
        return wikicode

def parse_dump(filepath, max_articles=None):
    NS = detect_namespace(filepath)
    articles = []
    count = 0
    t0 = time.time()

    print('Parseando dump XML...')
    with bz2.open(filepath, 'rt', encoding='utf-8') as f:
        for event, elem in ET.iterparse(f, events=('end',)):
            if elem.tag != f'{NS}page':
                continue

            ns_tag = elem.find(f'{NS}ns')
            if ns_tag is None or ns_tag.text != '0':
                elem.clear()
                continue

            revision = elem.find(f'{NS}revision')
            if revision is None:
                elem.clear()
                continue

            text_elem = revision.find(f'{NS}text')
            wikitext = text_elem.text if (text_elem is not None and text_elem.text) else ''

            if wikitext.strip().upper().startswith('#REDIRECT'):
                elem.clear()
                continue

            title_elem = elem.find(f'{NS}title')
            id_elem    = elem.find(f'{NS}id')
            ts_elem    = revision.find(f'{NS}timestamp')

            plain_text = wikitext
            categories = re.findall(r'\[\[Category:([^\]|]+)', wikitext)
            num_links  = len(re.findall(r'\[\[[^:][^\]]*\]\]', wikitext))
            num_refs   = len(re.findall(r'<ref[\s>]', wikitext))
            num_sections = len(re.findall(r'^==[^=]', wikitext, re.MULTILINE))

            articles.append({
                'id':             int(id_elem.text) if id_elem is not None else 0,
                'title':          title_elem.text if title_elem is not None else '',
                'word_count':     len(plain_text.split()),
                'text_length':    len(plain_text),
                'timestamp':      ts_elem.text if ts_elem is not None else '',
                'num_categories': len(categories),
                'num_links':      num_links,
                'num_references': num_refs,
                'num_sections':   num_sections,
            })

            count += 1
            if count % 5000 == 0:
                print(f'  {count:>6,} artículos  ({time.time()-t0:.0f}s)')
            elem.clear()

            if max_articles and count >= max_articles:
                break

    print(f'Terminado: {count:,} artículos  ({time.time()-t0:.0f}s)')
    return articles

In [10]:
for file in dump_files:

    print(f"\nProcesando {file}")

    articles_part = parse_dump(file, max_articles=40000)

    df_part = spark.createDataFrame(articles_part)

    df_part = df_part.withColumn(
        "source_dump",
        F.lit(file)
    )

    output_path = f"wiki_parquet/{file.split('/')[-1]}"

    df_part.write.mode("overwrite").parquet(output_path)


Procesando data/wikipedia_src/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2
Namespace: {http://www.mediawiki.org/xml/export-0.11/}
Parseando dump XML...
   5,000 artículos  (33s)
  10,000 artículos  (67s)
  15,000 artículos  (99s)
  20,000 artículos  (123s)
Terminado: 21,009 artículos  (127s)

Procesando data/wikipedia_src/enwiki-20260201-pages-articles-multistream2.xml-p41243p151573.bz2
Namespace: {http://www.mediawiki.org/xml/export-0.11/}
Parseando dump XML...
   5,000 artículos  (23s)
  10,000 artículos  (47s)
  15,000 artículos  (70s)
  20,000 artículos  (87s)
  25,000 artículos  (103s)
  30,000 artículos  (117s)
  35,000 artículos  (128s)
  40,000 artículos  (137s)
Terminado: 40,000 artículos  (137s)

Procesando data/wikipedia_src/enwiki-20260201-pages-articles-multistream3.xml-p151574p311329.bz2
Namespace: {http://www.mediawiki.org/xml/export-0.11/}
Parseando dump XML...
   5,000 artículos  (19s)
  10,000 artículos  (39s)
  15,000 artículos  (56s)
  20,000 artícu

In [11]:
df = spark.read.parquet("wiki_parquet/*")


In [12]:
df.groupBy("source_dump").count().show()

+--------------------+-----+
|         source_dump|count|
+--------------------+-----+
|data/wikipedia_sr...|40000|
|data/wikipedia_sr...|40000|
|data/wikipedia_sr...|21009|
+--------------------+-----+



## Filtrar:

In [13]:
df.filter(F.col("word_count") > 2000).show()

+------+--------------+---------+--------------+------------+-----------+--------------------+--------------------+----------+--------------------+
|    id|num_categories|num_links|num_references|num_sections|text_length|           timestamp|               title|word_count|         source_dump|
+------+--------------+---------+--------------+------------+-----------+--------------------+--------------------+----------+--------------------+
|151575|            11|      427|           307|          11|     159868|2026-01-31T14:50:27Z|      Cathay Pacific|     14093|data/wikipedia_sr...|
|151577|             5|       71|             9|          10|      15792|2026-01-07T22:31:51Z| Causality (physics)|      2125|data/wikipedia_sr...|
|151578|             4|      231|            72|          16|      43569|2026-01-11T13:00:26Z|            Wetherby|      5289|data/wikipedia_sr...|
|151579|            45|      316|           130|          10|      53880|2026-01-16T13:16:45Z|         Bill Bail

## Columna nueva:

In [14]:
df = df.withColumn(
    "refs_per_1000",
    F.when(
        F.col("word_count") > 0,
        F.col("num_references") / F.col("word_count") * 1000
    ).otherwise(None)
)

In [15]:
df['refs_per_1000']

Column<'refs_per_1000'>

## Estadísticas

In [16]:
df.describe().show()

+-------+------------------+------------------+------------------+------------------+-----------------+------------------+--------------------+------------------+-----------------+--------------------+------------------+
|summary|                id|    num_categories|         num_links|    num_references|     num_sections|       text_length|           timestamp|             title|       word_count|         source_dump|     refs_per_1000|
+-------+------------------+------------------+------------------+------------------+-----------------+------------------+--------------------+------------------+-----------------+--------------------+------------------+
|  count|            101009|            101009|            101009|            101009|           101009|            101009|              101009|            101009|           101009|              101009|            101009|
|   mean|118691.87500123751| 7.823847379936441| 170.4918076607035|46.123355344573255|7.724984902335436|32809.6990070

## Agrupación

In [17]:
df.groupBy("source_dump").avg("word_count").show()

+--------------------+-----------------+
|         source_dump|  avg(word_count)|
+--------------------+-----------------+
|data/wikipedia_sr...|       3218.96845|
|data/wikipedia_sr...|      3317.398025|
|data/wikipedia_sr...|5634.064258175068|
+--------------------+-----------------+

